# JackSparrow v16 — BTC Perp ML Trading System

**v16 upgrades:** `ModelRegistry` · `RetrainEngine` · lightweight warm-start · auto-apply adaptive updates · `hot_reload_model()` · F1+WinRate validation gate

> Run cells in order. Set `RETRAIN_LIGHTWEIGHT = True` in Cell 5 for fast scheduled retrains.

In [ ]:
# Cell 1: Setup — JackSparrow v16
# GitHub: https://github.com/energyforreal/JackSparrow
#
# ═══════════════════════════════════════════════════════
#  v16 ARCHITECTURE UPGRADES (over v15)
# ═══════════════════════════════════════════════════════
#  [v16-1] ModelRegistry class — versioned save/load/hot-swap
#  [v16-2] RetrainEngine class — modular, callable retrain logic
#  [v16-3] Lightweight retrain mode (warm-start, no CV) for speed
#  [v16-4] Auto-apply adaptive updates to training_results
#  [v16-5] hot_reload_model() helper for live agent integration
#  [v16-6] Retrain cooldown tracked via registry (not just log file)
#  [v16-7] Incremental data fetch helper (latest N bars only)
#  [v16-8] Validation gate improved: also checks win-rate delta
#
# v15 fixes retained (unchanged):
#  Strict 80/20 temporal holdout | per-fold sanitise+corr-prune |
#  train-derived median imputation | ffill(limit=5) only | frozen
#  feature order | tighter KS drift alpha | position-scaled cost |
#  SELL/BUY weight boost | ATR trailing | edge decay | vol filter

import subprocess, sys

pkgs = [
    "xgboost==2.0.2", "scikit-learn", "pandas", "numpy",
    "requests", "joblib", "matplotlib", "seaborn", "shap",
    "tqdm", "numba", "imbalanced-learn", "scipy",
]
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)
except subprocess.CalledProcessError as e:
    print(f"[WARN] pip install partial failure: {e}")

import requests, time, json, joblib, shap, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy
from scipy.stats import ks_2samp
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import clone
import xgboost as xgb
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped (not in Colab)')

BASE_URL   = "https://api.india.delta.exchange/v2"
SYMBOL     = "BTCUSD"
PRODUCT_ID = 27
TIMEFRAMES = ["5m", "15m"]

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/JackSparrow_Models")
    if Path("/content/drive").exists()
    else Path("./models")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTRACT_VALUE_BTC = 0.001
TAKER_FEE          = 0.0005
SLIPPAGE_BPS       = 5

TP_LONG_PCT  = 0.006
SL_LONG_PCT  = 0.004
TP_SHORT_PCT = 0.004
SL_SHORT_PCT = 0.006

RANDOM_STATE = 42

# ── Signal quality thresholds ─────────────────────────────────────────────────
CONFIDENCE_PERCENTILE = 90
EDGE_FLOOR            = 0.15

# ── Execution optimisations ───────────────────────────────────────────────────
ATR_TRAILING_MULT      = 1.5
MIN_HOLD_BARS          = 5
EDGE_DECAY_THRESHOLD   = 0.05
VOLATILITY_FILTER_PCT  = 0.002

# ── Trade management ──────────────────────────────────────────────────────────
MIN_EDGE_COST_RATIO   = 2.0
MIN_GAP_CANDLES       = 3
TF_MAX_TRADES_DAY     = {"5m": 8, "15m": 4}

# ── Feature selection ─────────────────────────────────────────────────────────
TOP_N_FEATURES   = 20
CORR_THRESHOLD   = 0.95
HOLDOUT_FRAC     = 0.20

# ── v16: Retrain / lifecycle config ──────────────────────────────────────────
ROLLING_DAYS                = 60
DRIFT_FEATURE_LIMIT         = 5
DRIFT_ALPHA                 = 0.01
DRIFT_STAT_THRESHOLD        = 0.10
RETRAIN_COOLDOWN_HOURS      = 12
VALIDATION_WINDOW_ROWS      = 10_000
MIN_VAL_ROWS                = 200
NUM_BOOST_ROUND_INCREMENTAL = 150
CLASS_WEIGHTS               = {0: 1.3, 1: 0.5, 2: 1.3}

print("Setup complete (v16)")
print(f"   Active timeframes       : {TIMEFRAMES}")
print(f"   Output directory        : {OUTPUT_DIR}")
print(f"   TP long / short         : {TP_LONG_PCT:.1%} / {TP_SHORT_PCT:.1%}")
print(f"   SL long / short         : {SL_LONG_PCT:.1%} / {SL_SHORT_PCT:.1%}")
print(f"   Signal threshold        : top {100-CONFIDENCE_PERCENTILE}% by |edge|, floor={EDGE_FLOOR}")
print(f"   ATR trailing multiplier : {ATR_TRAILING_MULT}x")
print(f"   Min hold bars           : {MIN_HOLD_BARS}")
print(f"   Edge decay threshold    : {EDGE_DECAY_THRESHOLD}")
print(f"   Volatility filter       : atr_pct >= {VOLATILITY_FILTER_PCT}")
print(f"   Top-N features          : {TOP_N_FEATURES}")


In [ ]:
# Cell 2: Fetch historical OHLCV — v10 CRITICAL FIX + [v16-7] incremental fetch
#
# ROOT CAUSE OF THE 2100-ROW BUG (fixed in v10):
# Delta Exchange /v2/history/candles returns candles DESCENDING (newest first).
# v9 loop used batch[-1]["time"] + resolution_s  → barely advanced → duplicates.
#
# FIX [v10-1/v10-2]: Walk BACKWARD; reverse at end for ascending order.
# FIX [v10-3]: Hard progress guard — force window jump if stuck.
# FIX [v10-4]: Debug logging on every request.
# [v16-7]: fetch_latest_candles() — fetch only the most recent N bars
#          (used by RetrainEngine to avoid re-downloading full history).

import requests, time
import pandas as pd
import numpy as np

try:
    from tqdm.auto import tqdm as _tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

MAX_CANDLES_PER_REQ = 2000
REQUEST_DELAY       = 0.12
MAX_RETRIES         = 3
FFILL_GAP_LIMIT     = 3
TF_TO_SECONDS       = {"5m": 300, "15m": 900}
TF_TARGET_CANDLES   = {"5m": 200_000, "15m": 100_000}
GAP_BUFFER_RATIO    = 1.10
OHLCV_COLS          = ["open", "high", "low", "close", "volume"]
HEADERS             = {"User-Agent": "python-rest-client/1.0", "Accept": "application/json"}


def fetch_candles_delta_v10(symbol, resolution, target_candles):
    all_candles  = []
    resolution_s = TF_TO_SECONDS[resolution]
    now_ts       = int(time.time())
    current_end  = now_ts - resolution_s
    buffer_candles = int(target_candles * GAP_BUFFER_RATIO)
    hard_start_ts  = current_end - (buffer_candles * resolution_s)
    window_size    = MAX_CANDLES_PER_REQ * resolution_s
    _req_count     = 0

    pbar = (
        _tqdm(total=target_candles, desc=f"{symbol} {resolution}",
              unit="candle", dynamic_ncols=True, miniters=500)
        if HAS_TQDM else None
    )
    _prev_pbar_n = 0

    def _update_bar(done=False):
        nonlocal _prev_pbar_n
        if pbar is None:
            return
        new_n = min(len(all_candles), target_candles)
        delta = new_n - _prev_pbar_n
        if delta > 0:
            pbar.update(delta)
            _prev_pbar_n = new_n
        cur_date = pd.to_datetime(current_end, unit="s").strftime("%Y-%m-%d")
        pbar.set_postfix_str(f"window_end={cur_date}  reqs={_req_count}", refresh=False)
        if done:
            pbar.n = target_candles
            pbar.set_postfix_str(f"done  reqs={_req_count}", refresh=True)
            pbar.close()

    while len(all_candles) < target_candles and current_end > hard_start_ts:
        current_start = max(current_end - window_size, hard_start_ts)
        params = {
            "symbol": symbol, "resolution": resolution,
            "start": int(current_start), "end": int(current_end),
        }
        prev_len = len(all_candles)
        for retry in range(MAX_RETRIES):
            try:
                r = requests.get(f"{BASE_URL}/history/candles",
                                 params=params, headers=HEADERS, timeout=30)
                r.raise_for_status()
                data = r.json()
                if not data.get("success"):
                    raise ValueError(f"API error: {data.get('error', data)}")
                batch = data.get("result", [])
                _req_count += 1
                if batch:
                    newest = pd.to_datetime(batch[0]["time"],  unit="s").strftime("%Y-%m-%d %H:%M")
                    oldest = pd.to_datetime(batch[-1]["time"], unit="s").strftime("%Y-%m-%d %H:%M")
                    print(f"  [{resolution}] req#{_req_count:04d}: {len(batch):4d} candles"
                          f" | {oldest} -> {newest}"
                          f" | total: {len(all_candles) + len(batch):,}")
                else:
                    s_str = pd.to_datetime(current_start, unit="s").strftime("%Y-%m-%d")
                    e_str = pd.to_datetime(current_end,   unit="s").strftime("%Y-%m-%d")
                    print(f"  [{resolution}] req#{_req_count:04d}: EMPTY | window {s_str} -> {e_str}")
                if not batch:
                    current_end = current_start - resolution_s
                    _update_bar()
                    break
                all_candles.extend(batch)
                current_end = batch[-1]["time"] - resolution_s
                _update_bar()
                if len(all_candles) >= target_candles:
                    _update_bar(done=True)
                    return all_candles[:target_candles]
                time.sleep(REQUEST_DELAY)
                break
            except Exception as e:
                if retry == MAX_RETRIES - 1:
                    print(f"  Error after {MAX_RETRIES} retries: {e}")
                    if pbar: pbar.close()
                    return all_candles[:target_candles]
                time.sleep(2 ** retry)
        if len(all_candles) == prev_len:
            print(f"  [{resolution}] No progress -- forcing window jump")
            current_end -= window_size

    if pbar:
        _update_bar(done=True)
    return all_candles[:target_candles]


# [v16-7] Incremental fetch: only retrieve the latest `n_bars` candles.
# Used by RetrainEngine to extend the dataset without full re-download.
def fetch_latest_candles(symbol, resolution, n_bars=5000):
    """Fetch only the most recent n_bars candles (incremental update)."""
    resolution_s = TF_TO_SECONDS[resolution]
    now_ts       = int(time.time())
    end_ts       = now_ts - resolution_s
    start_ts     = end_ts - (n_bars * resolution_s * 1.05)  # 5% buffer
    params = {
        "symbol": symbol, "resolution": resolution,
        "start": int(start_ts), "end": int(end_ts),
    }
    for retry in range(MAX_RETRIES):
        try:
            r = requests.get(f"{BASE_URL}/history/candles",
                             params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            data = r.json()
            if not data.get("success"):
                raise ValueError(f"API error: {data.get('error', data)}")
            candles = data.get("result", [])
            candles = list(reversed(candles))  # ascending: oldest ... newest
            out = candles[-n_bars:]  # latest n bars (head would be oldest)
            print(f"  [incremental:{resolution}] fetched {len(candles):,} candles → latest {len(out):,}")
            return out
        except Exception as e:
            if retry == MAX_RETRIES - 1:
                print(f"  [incremental:{resolution}] Error: {e}")
                return []
            time.sleep(2 ** retry)
    return []


def validate_and_clean(df, resolution):
    resolution_s = TF_TO_SECONDS[resolution]
    report = {}
    df = df.sort_values("time").reset_index(drop=True)
    for col in OHLCV_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    n_dups = df.duplicated(subset=["time"]).sum()
    if n_dups > 0:
        print(f"  [{resolution}] {n_dups} duplicate timestamps removed.")
    df = df.drop_duplicates(subset=["time"]).reset_index(drop=True)
    report["duplicates_removed"] = int(n_dups)
    df["_delta"] = df["time"].diff()
    gap_mask     = (df["_delta"] != resolution_s) & df["_delta"].notna()
    n_raw_gaps   = int(gap_mask.sum())
    df["datetime"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.set_index("datetime")
    freq_alias = f"{resolution_s}s"
    full_idx   = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq_alias, tz="UTC")
    df           = df.reindex(full_idx)
    missing_mask = df["time"].isna()
    n_missing    = int(missing_mask.sum())
    run_id   = (missing_mask != missing_mask.shift()).cumsum()
    run_lens = missing_mask.groupby(run_id).transform("sum")
    large_gap = missing_mask & (run_lens > FFILL_GAP_LIMIT)
    df = df.ffill()
    df.loc[large_gap, OHLCV_COLS] = np.nan
    df["time"]     = (df.index.astype("int64") // 1_000_000_000).astype("int64")
    df["datetime"] = df.index
    df = df.reset_index(drop=True).drop(columns=["_delta"], errors="ignore")
    now_ts = int(time.time())
    df = df[df["time"] <= now_ts - resolution_s].reset_index(drop=True)
    report.update({
        "raw_gaps":      n_raw_gaps,
        "missing_bars":  n_missing,
        "final_candles": len(df),
        "date_start":    str(df["datetime"].min().date()) if len(df) else "N/A",
        "date_end":      str(df["datetime"].max().date()) if len(df) else "N/A",
    })
    return df, report


def print_quality_report(tf, report, target):
    n      = report["final_candles"]
    pct    = n / target * 100 if target else 0
    status = "OK" if pct >= 90 else "WARN"
    print(f"  [{status}] {tf}: {n:,} / {target:,} ({pct:.1f}%) | "
          f"{report['date_start']} -> {report['date_end']} | "
          f"dups removed: {report['duplicates_removed']}")


raw_data = {}
for tf in TIMEFRAMES:
    target = TF_TARGET_CANDLES[tf]
    print(f"\n{'='*60}")
    print(f"Fetching {tf} | target={target:,} | buffer={int(target*GAP_BUFFER_RATIO):,}")
    candles = fetch_candles_delta_v10(SYMBOL, tf, target)
    if not candles:
        print(f"  {tf}: no candles returned -- skipping.")
        continue
    candles = list(reversed(candles))
    df = pd.DataFrame(candles)
    df["datetime"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df, report = validate_and_clean(df, tf)
    raw_data[tf] = df
    print_quality_report(tf, report, target)

print("\nAll fetches complete.")


In [ ]:
# Cell 2.5 Debug — verify raw_data shape (target: ~200k / ~100k rows)
print("=== RAW_DATA DEBUG ===")
print(f"Keys: {list(raw_data.keys())}")
for tf in ["5m", "15m"]:
    if tf in raw_data:
        df = raw_data[tf]
        target = {"5m": 200_000, "15m": 100_000}[tf]
        pct = len(df) / target * 100
        status = "PASS" if pct >= 90 else "FAIL"
        print(f"\n{tf}:")
        print(f"  Shape       : {df.shape}")
        print(f"  Coverage    : {pct:.1f}% of target {target:,}  [{status}]")
        print(f"  Date range  : {df['datetime'].min()} -> {df['datetime'].max()}")
        print(f"  Sample:")
        print(df[["datetime","open","high","low","close","volume"]].head(3).to_string())
    else:
        print(f"\n{tf}: NOT PRESENT -- fetch failed, fix Cell 2 first")
print("\n=== END DEBUG ===")


In [ ]:
# Cell 3: Feature Engineering + Sanitisation Pipeline (v12, unchanged)
# [v11-1]  54-indicator suite (retained)
# [v11-3]  250-bar warmup (retained)
# [v12-3]  sanitize_features, remove_correlated_features helpers
# [v16]    No changes — this cell is already production-ready.

def prepare_base_df(df):
    df = df.copy().sort_values("datetime").reset_index(drop=True)
    for col in ["open","high","low","close","volume"]:
        df[col] = df[col].astype(float)
    return df


def build_features(df):
    out = df.copy()
    c, h, l, o, v = (out[x].astype(float) for x in ["close","high","low","open","volume"])
    eps = 1e-10

    out["returns"]     = c.pct_change(fill_method=None)
    out["log_returns"] = np.log(c / c.shift(1))
    out["roc_5"]  = c.pct_change(5,  fill_method=None)
    out["roc_10"] = c.pct_change(10, fill_method=None)
    out["roc_20"] = c.pct_change(20, fill_method=None)

    ema9   = c.ewm(span=9,   adjust=False).mean()
    ema21  = c.ewm(span=21,  adjust=False).mean()
    ema50  = c.ewm(span=50,  adjust=False).mean()
    ema100 = c.ewm(span=100, adjust=False).mean()
    ema200 = c.ewm(span=200, adjust=False).mean()
    out["ema_9"]            = ema9
    out["ema_21"]           = ema21
    out["ema_50"]           = ema50
    out["ema_100"]          = ema100
    out["ema_200"]          = ema200
    out["ema_cross_9_21"]   = np.sign(ema9  - ema21)
    out["ema_cross_21_50"]  = np.sign(ema21 - ema50)
    out["ema_cross_50_200"] = np.sign(ema50 - ema200)
    out["price_vs_ema200"]  = (c - ema200) / (ema200 + eps)

    macd     = c.ewm(span=12,adjust=False).mean() - c.ewm(span=26,adjust=False).mean()
    macd_sig = macd.ewm(span=9, adjust=False).mean()
    out["macd"]        = macd
    out["macd_signal"] = macd_sig
    out["macd_hist"]   = macd - macd_sig

    def _rsi(s, p):
        d = s.diff()
        ag = d.clip(lower=0).ewm(com=p-1, adjust=False).mean()
        al = (-d).clip(lower=0).ewm(com=p-1, adjust=False).mean()
        return 100.0 - 100.0 / (1.0 + ag / (al + eps))
    out["rsi_14"] = _rsi(c, 14)
    out["rsi_7"]  = _rsi(c, 7)

    pc  = c.shift(1)
    tr  = pd.concat([(h-l), (h-pc).abs(), (l-pc).abs()], axis=1).max(axis=1)
    atr = tr.ewm(com=13, adjust=False).mean()
    out["atr_14"]  = atr
    out["atr_pct"] = atr / (c + eps)

    ret = out["returns"]
    out["volatility"]    = ret.rolling(20).std()
    out["volatility_10"] = ret.rolling(10).std()
    out["volatility_20"] = ret.rolling(20).std()

    sma20, std20 = c.rolling(20).mean(), c.rolling(20).std()
    bb_u, bb_l = sma20 + 2*std20, sma20 - 2*std20
    bb_rng = bb_u - bb_l + eps
    out["bb_width"] = bb_rng / (sma20 + eps)
    out["bb_pct"]   = (c - bb_l) / bb_rng

    hd, ld = h.diff(), -l.diff()
    pdm = pd.Series(np.where((hd>ld)&(hd>0), hd.fillna(0), 0.0), index=out.index)
    mdm = pd.Series(np.where((ld>hd)&(ld>0), ld.fillna(0), 0.0), index=out.index)
    pdi = 100 * pdm.ewm(com=13,adjust=False).mean() / (atr+eps)
    mdi = 100 * mdm.ewm(com=13,adjust=False).mean() / (atr+eps)
    dx  = 100 * (pdi-mdi).abs() / (pdi+mdi+eps)
    adx = dx.ewm(com=13, adjust=False).mean()
    out["plus_di"]         = pdi
    out["minus_di"]        = mdi
    out["di_diff"]         = pdi - mdi
    out["adx_14"]          = adx
    out["regime_trending"] = (adx > 25.0).astype(float)
    out["regime_ranging"]  = (adx <= 25.0).astype(float)

    low14, high14 = l.rolling(14).min(), h.rolling(14).max()
    sk = 100 * (c - low14) / (high14 - low14 + eps)
    out["stoch_k"] = sk
    out["stoch_d"] = sk.rolling(3).mean()

    tp  = (h + l + c) / 3
    mad = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    out["cci"] = (tp - tp.rolling(20).mean()) / (0.015 * mad + eps)

    vsma = v.rolling(20).mean()
    vstd = v.rolling(20).std()
    out["vol_zscore"] = (v - vsma) / (vstd + eps)
    out["vol_ratio"]  = v / (vsma + eps)

    full_rng = h - l + eps
    body     = (c - o).abs()
    hi_oc    = pd.concat([o, c], axis=1).max(axis=1)
    lo_oc    = pd.concat([o, c], axis=1).min(axis=1)
    out["hl_range"]           = h - l
    out["body"]               = body
    out["cdl_body_ratio"]     = body / full_rng
    out["cdl_upper_wick"]     = (h - hi_oc) / full_rng
    out["cdl_lower_wick"]     = (lo_oc - l) / full_rng
    out["cdl_body_direction"] = np.sign(c - o)

    high20, low20 = h.rolling(20).max(), l.rolling(20).min()
    sr_rng = high20 - low20 + eps
    out["sr_range_position"] = (c - low20) / sr_rng
    out["sr_near_high"]      = ((high20 - c) / (c+eps) < 0.005).astype(float)
    out["sr_near_low"]       = ((c - low20) / (c+eps) < 0.005).astype(float)
    out["sr_breakout_up"]    = (c > high20.shift(1)).astype(float)
    out["sr_breakout_dn"]    = (c < low20.shift(1)).astype(float)

    # Placeholders — NOT in BASE_FEATURE_COLS, never fed to model
    out["basis"]               = 0.0
    out["basis_pct"]           = 0.0
    out["funding_rate"]        = pd.Series(0.0, index=out.index)
    out["funding_zscore"]      = 0.0
    out["funding_cumsum"]      = 0.0
    out["funding_long_unfav"]  = 0.0
    out["funding_short_unfav"] = 0.0
    out["open_interest"]       = 0.0
    out["oi_change"]           = 0.0
    out["oi_zscore"]           = 0.0

    return out


def sanitize_features(X_df):
    before = X_df.shape[1]
    X_df = X_df.loc[:, X_df.nunique() > 1]
    nan_ratio = X_df.isna().mean()
    X_df = X_df.loc[:, nan_ratio < 0.30]
    after = X_df.shape[1]
    if before != after:
        print(f"  [Sanitise] {before} -> {after} features "
              f"(removed {before - after} constant/high-NaN columns)")
    return X_df


def remove_correlated_features(X_df, threshold=0.95):
    corr  = X_df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    if to_drop:
        print(f"  [CorrPrune] Dropping {len(to_drop)} highly-correlated features")
    return X_df.drop(columns=to_drop, errors='ignore'), to_drop


def feature_stability_check(df_full, feature_cols, target_col="label",
                              n_chunks=3, std_threshold=0.08,
                              top_n_fallback=None):
    chunks = np.array_split(df_full, n_chunks)
    stable = []
    for col in feature_cols:
        scores = []
        for chunk in chunks:
            if len(chunk) < 200:
                continue
            corr = chunk[col].corr(chunk[target_col])
            if np.isnan(corr) or np.isinf(corr):
                continue
            scores.append(abs(corr))
        if len(scores) >= 2 and np.std(scores) <= std_threshold:
            stable.append(col)
    if len(stable) < 5:
        fallback_cols = feature_cols[:(top_n_fallback or len(feature_cols))]
        print(f"  [Stability] Only {len(stable)} stable features found — "
              f"fallback: keeping top {len(fallback_cols)} features")
        stable = fallback_cols
    else:
        print(f"  [Stability] {len(stable)}/{len(feature_cols)} features pass stability")
    return stable


def merge_timeframes(df_5m, df_15m):
    df_5m  = df_5m.set_index("datetime")
    df_15m = df_15m.set_index("datetime")
    aligned = df_15m.reindex(df_5m.index, method="ffill").ffill(limit=5)
    aligned = aligned.add_suffix("_15m")
    return df_5m.join(aligned).reset_index()


def clean_dataset(df, warmup=250):
    print(f"Before cleaning: {len(df)} rows")
    df = df.iloc[warmup:].copy().ffill(limit=5)
    df = df.dropna(subset=["close", "open", "high", "low", "volume"])
    print(f"After cleaning:  {len(df)} rows")
    return df


def validate_dataset(df, name):
    print(f"\n=== VALIDATION: {name} ===")
    print("Rows:", len(df))
    nan_total = df.isna().sum().sum()
    print("Total NaNs (features):", nan_total)
    if len(df) < 1000:
        raise ValueError("Too few rows after cleaning")
    print("Dataset clean and ready")


# Execute
featured_data = {}
df5  = prepare_base_df(raw_data["5m"])
df15 = prepare_base_df(raw_data["15m"])
df5_feat  = build_features(df5)
df15_feat = build_features(df15)
df15_clean = clean_dataset(df15_feat)
validate_dataset(df15_clean, "15m standalone")
featured_data["15m"] = df15_clean
df_merged = merge_timeframes(df5_feat, df15_feat)
df5_clean = clean_dataset(df_merged)
validate_dataset(df5_clean, "5m+15m merged")
featured_data["5m"] = df5_clean

_meta = {"time","datetime","open","high","low","close","volume"}
for tf in TIMEFRAMES:
    if tf in featured_data:
        nf = len([c for c in featured_data[tf].columns if c not in _meta])
        print(f"  {tf}: {len(featured_data[tf]):,} rows x {nf} computed columns")


In [ ]:
# Cell 4: Perp-aware label generation (unchanged from v9)

from numba import njit

TF_LOOKAHEAD   = {"5m": 16, "15m": 12}
MAX_LOOKAHEAD  = max(TF_LOOKAHEAD.values())
TF_BARS_PER_8H = {"5m": 96, "15m": 32}


@njit
def _label_loop(close, high, low, fund, n, lookahead,
                tp_long_lvl, sl_long_lvl, tp_short_lvl, sl_short_lvl,
                bars_per_8h_f, fee2):
    labels = np.ones(n, dtype=np.int64)
    for i in range(n - lookahead):
        entry    = close[i]
        long_tp  = entry * tp_long_lvl
        long_sl  = entry * sl_long_lvl
        short_tp = entry * tp_short_lvl
        short_sl = entry * sl_short_lvl
        long_result = short_result = 0
        cum_fund_long = cum_fund_short = 0.0
        for k in range(1, lookahead + 1):
            h = high[i + k]; l = low[i + k]; fr = fund[i + k]
            cum_fund_long  += (fr  if fr  > 0.0 else 0.0) / bars_per_8h_f
            cum_fund_short += (-fr if fr  < 0.0 else 0.0) / bars_per_8h_f
            if long_result == 0:
                if h >= long_tp:   long_result = 1
                elif l <= long_sl: long_result = -1
            if short_result == 0:
                if l <= short_tp:   short_result = 1
                elif h >= short_sl: short_result = -1
            if long_result != 0 and short_result != 0:
                break
        long_net  = (tp_long_lvl  - 1.0) - fee2 - cum_fund_long
        short_net = (1.0 - tp_short_lvl) - fee2 - cum_fund_short
        if long_result  == 1 and long_net  > 0.0: labels[i] = 2
        elif short_result == 1 and short_net > 0.0: labels[i] = 0
    return labels


def make_labels(df, tf):
    bars_per_8h = TF_BARS_PER_8H[tf]
    return pd.Series(
        _label_loop(
            df["close"].values, df["high"].values, df["low"].values,
            df["funding_rate"].fillna(0).values,
            len(df), TF_LOOKAHEAD[tf],
            1.0 + TP_LONG_PCT, 1.0 - SL_LONG_PCT,
            1.0 - TP_SHORT_PCT, 1.0 + SL_SHORT_PCT,
            float(bars_per_8h), TAKER_FEE * 2,
        ),
        index=df.index, dtype=int
    )


labeled_data   = {}
train_splits   = {}
holdout_splits = {}

for tf in TIMEFRAMES:
    df = featured_data.get(tf, pd.DataFrame())
    if df.empty:
        print(f"  {tf}: featured data missing -- skipping")
        continue
    df = df.copy()
    df["label"] = make_labels(df, tf)
    labeled_data[tf] = df
    split_idx             = int(len(df) * (1 - HOLDOUT_FRAC))
    train_splits[tf]      = df.iloc[:split_idx - MAX_LOOKAHEAD].copy()
    holdout_splits[tf]    = df.iloc[split_idx:].copy()
    labeled_data[tf]      = df
    counts = df["label"].value_counts()
    total  = len(df)
    sell_n = counts.get(0, 0)
    hold_n = counts.get(1, 0)
    buy_n  = counts.get(2, 0)
    print(f"\n{tf} label distribution ({total:,} samples):")
    print(f"  SELL (0): {sell_n:>7,}  ({sell_n/total*100:.1f}%)")
    print(f"  HOLD (1): {hold_n:>7,}  ({hold_n/total*100:.1f}%)")
    print(f"  BUY  (2): {buy_n:>7,}  ({buy_n/total*100:.1f}%)")
    bsr = buy_n / sell_n if sell_n > 0 else 0
    act = (sell_n + buy_n) / total * 100
    print(f"  BUY/SELL ratio: {bsr:.2f}  (target 0.5-2.0) {'OK' if 0.5<bsr<2.0 else 'WARN'}")
    print(f"  Actionable    : {act:.1f}%  (target >=25%)  {'OK' if act>=25 else 'WARN'}")
    print(f"  Train rows: {len(train_splits[tf]):,}  |  Holdout rows: {len(holdout_splits[tf]):,}")


In [ ]:
# Cell 5: Training — v16
#
# Architecture:
#   Phase 1 – Feature reduction  (sanitise → correlation prune)
#   Phase 2 – Cross-validation   (TimeSeriesSplit, XGBoost + balanced weights)
#   Phase 3 – Feature selection  (average importance → top-N → stability check)
#   Phase 4 – Final model        (retrain on ALL train data, selected features)
#
# [v16-3] RETRAIN_LIGHTWEIGHT mode:
#   - Skips CV entirely (Phase 2 skipped)
#   - Reuses existing feature selection from a previous pipeline
#   - Warm-starts the existing booster with NUM_BOOST_ROUND_INCREMENTAL trees
#   - Use this when RetrainEngine triggers a scheduled retrain, not a full retrain
#   - Set RETRAIN_LIGHTWEIGHT = False for initial / periodic full retrains

RETRAIN_LIGHTWEIGHT = False  # [v16-3] Set True for fast warm-start retrain

BASE_FEATURE_COLS = [
    "returns","log_returns","volatility",
    "ema_9","ema_21","ema_50","ema_100","ema_200",
    "ema_cross_9_21","ema_cross_21_50","ema_cross_50_200","price_vs_ema200",
    "macd","macd_signal","macd_hist",
    "rsi_14","rsi_7",
    "atr_14","atr_pct",
    "bb_width","bb_pct",
    "plus_di","minus_di","di_diff","adx_14",
    "regime_trending","regime_ranging",
    "vol_zscore","vol_ratio",
    "roc_5","roc_10","roc_20",
    "cdl_body_ratio","cdl_upper_wick","cdl_lower_wick","cdl_body_direction",
    "sr_range_position","sr_near_high","sr_near_low","sr_breakout_up","sr_breakout_dn",
    "stoch_k","stoch_d","cci",
    "volatility_10","volatility_20","hl_range",
]
CTX_COLS = [f"{col}_15m" for col in
            ["ema_cross_50_200","adx_14","regime_trending","regime_ranging",
             "rsi_14","di_diff","macd_hist"]]

N_SPLITS          = 5
MIN_TRAIN_ROWS    = 1_000
MIN_FOLD_PRESENCE = 3


def build_xgb():
    return XGBClassifier(
        n_estimators     = 200,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.80,
        colsample_bytree = 0.80,
        eval_metric      = "mlogloss",
        random_state     = RANDOM_STATE,
        n_jobs           = -1,
        verbosity        = 0,
    )


# [v16-3] Lightweight warm-start retrain (no CV, no feature re-selection)
def _lightweight_retrain(tf, df_clean, stable_features, old_model):
    """
    Warm-start the existing XGBoost booster with recent data.
    Skips CV and feature re-selection — uses frozen feature set from full train.
    Returns new_model (sklearn-compatible wrapper) and updated train_median.
    """
    print(f"  [v16-3:LIGHTWEIGHT] Warm-start retrain for {tf}")
    train_median_series = df_clean[stable_features].median()
    X_vals = df_clean[stable_features].fillna(train_median_series).values
    y_vals = df_clean["label"].values
    sw = np.array([CLASS_WEIGHTS.get(int(lbl), 1.0) for lbl in y_vals])

    old_booster   = old_model.get_booster()
    native_params = dict(old_model.get_xgb_params())
    native_params["num_class"] = 3

    dtrain = xgb.DMatrix(X_vals, label=y_vals, weight=sw)
    new_booster = xgb.train(
        native_params,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND_INCREMENTAL,
        xgb_model=old_booster,
        verbose_eval=False,
    )
    new_model = clone(old_model)
    new_model._Booster = new_booster
    for attr in ("classes_", "_le", "n_features_in_", "_features_count"):
        if hasattr(old_model, attr):
            try: setattr(new_model, attr, getattr(old_model, attr))
            except Exception: pass
    if not hasattr(new_model, "n_classes_"):
        new_model.n_classes_ = int(native_params.get("num_class", 3))

    f1 = f1_score(y_vals, new_model.predict(X_vals), average="macro", zero_division=0)
    print(f"  [v16-3:LIGHTWEIGHT] Train F1={f1:.4f}  Samples={len(y_vals):,}")
    return new_model, train_median_series, f1


training_results = {}

for tf in TIMEFRAMES:
    print(f"\n{'='*60}\nTraining: {tf}")

    if tf not in labeled_data or labeled_data[tf].empty:
        print(f"  No labeled data -- skipping")
        training_results[tf] = {"model": None, "scaler": None, "features": []}
        continue

    df = train_splits[tf].copy()
    print(f"  Train zone: {len(df):,} rows (holdout: {len(holdout_splits[tf]):,} rows reserved)")

    candidate    = BASE_FEATURE_COLS + (CTX_COLS if tf == "5m" else [])
    raw_feat_cols = [f for f in candidate if f in df.columns]

    df_clean = df[raw_feat_cols + ["label"]].copy()
    df_clean[raw_feat_cols] = (df_clean[raw_feat_cols]
                                .replace([np.inf, -np.inf], np.nan)
                                .ffill(limit=5))
    df_clean = df_clean.dropna(subset=["label"])

    print(f"  Rows after ffill: {len(df_clean):,}  |  Candidate features: {len(raw_feat_cols)}")

    if len(df_clean) < MIN_TRAIN_ROWS:
        print(f"  FAIL: only {len(df_clean):,} rows -- skipping")
        training_results[tf] = {"model": None, "scaler": None, "features": []}
        continue

    # ── [v16-3] Lightweight retrain path ─────────────────────────────────
    existing = training_results.get(tf, {})
    if RETRAIN_LIGHTWEIGHT and existing.get("model") is not None:
        existing_features = existing["features"]
        df_lw = df_clean[existing_features + ["label"]].dropna(subset=["label"])
        new_model, new_median, f1 = _lightweight_retrain(tf, df_lw, existing_features, existing["model"])
        training_results[tf].update({
            "model": new_model,
            "train_median": new_median,
            "mean_f1": f1,
        })
        print(f"  {tf} lightweight retrain complete (F1={f1:.4f})")
        continue

    # ── Full training path (Phase 2-4) ────────────────────────────────────
    X_all_vals = df_clean[raw_feat_cols].values
    y_all_vals = df_clean["label"].values

    tscv        = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_scores = []
    fold_imps   = []

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_all_vals)):
        Xtr_raw = pd.DataFrame(X_all_vals[tr_idx], columns=raw_feat_cols)
        Xva_raw = pd.DataFrame(X_all_vals[va_idx], columns=raw_feat_cols)
        ytr, yva = y_all_vals[tr_idx], y_all_vals[va_idx]
        if len(np.unique(ytr)) < 2:
            continue
        train_median = Xtr_raw.median()
        Xtr_imp = Xtr_raw.fillna(train_median)
        Xva_imp = Xva_raw.fillna(train_median)
        Xtr_san = sanitize_features(Xtr_imp)
        Xtr_pru, _ = remove_correlated_features(Xtr_san, threshold=CORR_THRESHOLD)
        fold_feature_cols = list(Xtr_pru.columns)
        Xva_pru = Xva_imp.reindex(columns=fold_feature_cols, fill_value=0)
        sw = np.array([CLASS_WEIGHTS.get(int(y), 1.0) for y in ytr])
        clf = build_xgb()
        clf.fit(Xtr_pru.values, ytr, sample_weight=sw)
        pred   = clf.predict(Xva_pru.values)
        f1_mac = f1_score(yva, pred, average="macro", zero_division=0)
        acc    = (pred == yva).mean()
        fold_scores.append(f1_mac)
        fold_imps.append((fold_feature_cols, clf.feature_importances_))
        print(f"    Fold {fold_idx+1}: F1={f1_mac:.4f}  Acc={acc:.4f}"
              f"  train={len(Xtr_pru):,}  val={len(Xva_pru):,}  feats={len(fold_feature_cols)}")

    if not fold_scores:
        print(f"  FAIL: no folds completed")
        training_results[tf] = {"model": None, "scaler": None, "features": []}
        continue

    mean_f1 = float(np.mean(fold_scores))

    imp_accum = {}
    fold_count = {}
    for fcols, fimps in fold_imps:
        for col, imp in zip(fcols, fimps):
            imp_accum[col]  = imp_accum.get(col, 0.0) + imp
            fold_count[col] = fold_count.get(col, 0) + 1

    valid_features = [col for col in imp_accum if fold_count[col] >= MIN_FOLD_PRESENCE]
    imp_series   = pd.Series({k: imp_accum[k] / fold_count[k] for k in valid_features}).sort_values(ascending=False)
    top_n        = min(TOP_N_FEATURES, len(imp_series))
    top_features = imp_series.head(top_n).index.tolist()

    print(f"\n  CV Mean F1 : {mean_f1:.4f}  (TimeSeriesSplit, no leakage)")
    print(f"  Top {top_n} features selected by average fold importance:")
    for feat, imp in imp_series.head(top_n).items():
        bar = "#" * int(imp / (imp_series.max() + 1e-12) * 20)
        print(f"    {feat:<35} {imp:.5f}  {bar}")

    stable_features = feature_stability_check(df_clean, top_features, target_col="label", top_n_fallback=top_n)
    stable_features = sorted(stable_features)
    if len(stable_features) < 10:
        stable_features = top_features[:10]

    df_final = df_clean[stable_features + ["label"]].copy().dropna(subset=["label"])
    train_final_median = df_final[stable_features].median()
    X_final = df_final[stable_features].fillna(train_final_median).values
    y_final = df_final["label"].values
    sw_final = np.array([CLASS_WEIGHTS.get(int(y), 1.0) for y in y_final])
    final_model = build_xgb()
    final_model.fit(X_final, y_final, sample_weight=sw_final)
    print(f"\n  Final model trained on {len(X_final):,} samples, {len(stable_features)} features")

    # Save — full pipeline dict
    save_dir = OUTPUT_DIR / f"model_{tf}_v16"
    save_dir.mkdir(parents=True, exist_ok=True)
    meta = {
        "model_name": f"jacksparrow_BTCUSD_{tf}", "symbol": SYMBOL,
        "timeframe": tf, "product_id": PRODUCT_ID, "model_version": "v16",
        "feature_count": len(stable_features), "features": stable_features,
        "train_median": train_final_median.to_dict(),
        "label_config": {
            "tp_long_pct": TP_LONG_PCT, "sl_long_pct": SL_LONG_PCT,
            "tp_short_pct": TP_SHORT_PCT, "sl_short_pct": SL_SHORT_PCT,
            "lookahead": TF_LOOKAHEAD[tf], "fee_rate": TAKER_FEE,
            "smote": False, "class_weights": CLASS_WEIGHTS,
        },
        "training": {
            "cv_strategy": "TimeSeriesSplit", "cv_f1_macro": round(mean_f1, 4),
            "n_final_samples": int(len(y_final)), "n_splits": N_SPLITS,
            "model_type": "XGBClassifier",
            "feature_selection": "per-fold-sanitise+corr-prune+importance+stability",
            "top_n_features": top_n,
            "holdout_frac": HOLDOUT_FRAC,
            "retrain_mode": "FULL",   # [v16-3]
        },
        "trained_at": pd.Timestamp.now("UTC").isoformat(),
    }
    pipeline = {
        "model": final_model,
        "features": stable_features,
        "train_median": train_final_median.to_dict(),
        "meta": meta,
    }
    pkl_path = save_dir / f"pipeline_{tf}_v16.pkl"
    with open(pkl_path, "wb") as fh:
        pickle.dump(pipeline, fh)
    # [v16-1] Write latest pointer immediately after full train
    with open(OUTPUT_DIR / f"pipeline_{tf}_latest.pkl", "wb") as fh:
        pickle.dump(pipeline, fh)
    with open(save_dir / f"metadata_BTCUSD_{tf}.json", "w") as fh:
        json.dump(meta, fh, indent=2)

    training_results[tf] = {
        "model": final_model,
        "scaler": None,
        "features": stable_features,
        "train_median": train_final_median,
        "dir": save_dir,
        "mean_f1": mean_f1,
        "importance": imp_series,
        "meta": meta,
    }
    print(f"  Saved: {save_dir}")
    print(f"  Latest pointer updated: pipeline_{tf}_latest.pkl")

    # Post-train drift check
    if len(df_final) >= 40000:
        df_recent_drift = df_final.iloc[-20000:]
        df_past_drift   = df_final.iloc[-40000:-20000]
        drifted_quick = []
        for col in stable_features:
            if col not in df_past_drift.columns or col not in df_recent_drift.columns:
                continue
            a = df_past_drift[col].dropna().values
            b = df_recent_drift[col].dropna().values
            if len(a) < 30 or len(b) < 30:
                continue
            stat, p = ks_2samp(a, b)
            if p < 0.01 and stat > 0.1:
                drifted_quick.append(col)
        if len(drifted_quick) > 5:
            print(f"  ⚠️  Drift detected in {len(drifted_quick)} features — Cell 8 retrain advised")
        else:
            print(f"  ✅  Drift check: {len(drifted_quick)} features drifted — stable")

    print(f"  {tf} training complete")


In [ ]:
# Cell 6: Filtered backtest — v15
# [v14-1] Runs on holdout_splits[tf] — never seen during training
# [v14-7] Trade cost scaled by abs(position) size
# [v15-3] ATR-based trailing stop (1.5 × ATR) replaces fixed trailing_pct
# [v15-4] min_hold_bars=5 prevents premature noise exits
# [v15-5] Edge-decay exit: close long/short when edge fades below threshold
# [v15-6] Volatility filter: skip trades when market is too quiet (atr_pct low)
#
# Signal: edge = p_buy - p_sell
#   Long  when edge >  rolling_threshold AND edge >= EDGE_FLOOR
#   Short when edge < -rolling_threshold AND edge <= -EDGE_FLOOR
# Regime filter: trades only in ranging market (ADX <= 25)

COST_PER_TRADE           = TAKER_FEE + (SLIPPAGE_BPS / 10_000)
TF_FUNDING_INTERVAL_BARS = {"5m": 96, "15m": 32}
TF_BARS_PER_YEAR         = {"5m": 105_120, "15m": 35_040}


def run_backtest(df, signal, price_col="close",
                 atr_arr=None,          # [v15-3] ATR array for dynamic trailing stop
                 atr_mult=1.5,          # [v15-3] ATR multiplier (1.5× ATR default)
                 hard_sl_pct=0.01,
                 cost_pct=0.0005,
                 edge_arr=None,         # [v15-5] edge array for decay-based exit
                 edge_decay_thr=0.05,   # [v15-5] exit when |edge| drops below this
                 min_hold_bars=5):      # [v15-4] minimum bars before allowing exit

    price = df[price_col].values
    n     = len(price)

    # Fallback: fixed 0.4% trailing if no ATR provided
    _fallback_trail = 0.004

    position    = 0        # 0=flat, 1=long, -1=short
    entry_price = 0.0
    stop_price  = 0.0
    entry_idx   = None

    equity = [1.0]
    trades = []

    for i in range(1, n):
        p   = price[i]
        sig = signal.iloc[i]

        # ATR-based stop distance for this bar [v15-3]
        if atr_arr is not None and not np.isnan(atr_arr[i]) and atr_arr[i] > 0:
            trail_dist = atr_mult * atr_arr[i]
        else:
            trail_dist = p * _fallback_trail

        # Current |edge| for decay check [v15-5]
        cur_edge = abs(edge_arr[i]) if edge_arr is not None else 1.0

        prev_equity = equity[-1]
        exit_flag   = False

        # ── 1. EXIT CONDITIONS ─────────────────────────────────────────────
        if position == 1:
            # Ratchet trailing stop upward [v15-3]
            stop_price = max(stop_price, p - trail_dist)

            bars_held  = i - entry_idx
            min_hold_ok = bars_held >= min_hold_bars  # [v15-4]

            # Conditions that can close the position
            hit_trail = p <= stop_price
            hit_hard  = p <= entry_price * (1 - hard_sl_pct)
            edge_gone = cur_edge < edge_decay_thr and min_hold_ok  # [v15-5]

            if min_hold_ok and (hit_trail or hit_hard or edge_gone):
                pnl = (p - entry_price) / entry_price - cost_pct
                trades.append({
                    "type": "LONG", "entry": entry_price, "exit": p,
                    "pnl": pnl, "bars": bars_held,
                    "exit_reason": ("trail" if hit_trail else
                                    "hard_sl" if hit_hard else "edge_decay"),
                })
                position  = 0
                exit_flag = True

        elif position == -1:
            # Ratchet trailing stop downward [v15-3]
            stop_price = min(stop_price, p + trail_dist)

            bars_held   = i - entry_idx
            min_hold_ok = bars_held >= min_hold_bars  # [v15-4]

            hit_trail = p >= stop_price
            hit_hard  = p >= entry_price * (1 + hard_sl_pct)
            edge_gone = cur_edge < edge_decay_thr and min_hold_ok  # [v15-5]

            if min_hold_ok and (hit_trail or hit_hard or edge_gone):
                pnl = (entry_price - p) / entry_price - cost_pct
                trades.append({
                    "type": "SHORT", "entry": entry_price, "exit": p,
                    "pnl": pnl, "bars": bars_held,
                    "exit_reason": ("trail" if hit_trail else
                                    "hard_sl" if hit_hard else "edge_decay"),
                })
                position  = 0
                exit_flag = True

        # ── 2. ENTRY / FLIP LOGIC ─────────────────────────────────────────
        if position == 0:
            if sig == 2:    # BUY signal
                position    = 1
                entry_price = p
                stop_price  = p - trail_dist
                entry_idx   = i
            elif sig == 0:  # SELL signal
                position    = -1
                entry_price = p
                stop_price  = p + trail_dist
                entry_idx   = i
        else:
            # Allow position flip (incurs cost)
            if position == 1 and sig == 0 and (i - entry_idx) >= min_hold_bars:
                pnl = (p - entry_price) / entry_price - cost_pct
                trades.append({
                    "type": "LONG", "entry": entry_price, "exit": p,
                    "pnl": pnl, "bars": i - entry_idx, "exit_reason": "flip",
                })
                position    = -1
                entry_price = p
                stop_price  = p + trail_dist
                entry_idx   = i

            elif position == -1 and sig == 2 and (i - entry_idx) >= min_hold_bars:
                pnl = (entry_price - p) / entry_price - cost_pct
                trades.append({
                    "type": "SHORT", "entry": entry_price, "exit": p,
                    "pnl": pnl, "bars": i - entry_idx, "exit_reason": "flip",
                })
                position    = 1
                entry_price = p
                stop_price  = p - trail_dist
                entry_idx   = i

        # ── 3. EQUITY UPDATE ─────────────────────────────────────────────
        if position == 1:
            ret = (price[i] - price[i-1]) / price[i-1]
        elif position == -1:
            ret = (price[i-1] - price[i]) / price[i-1]
        else:
            ret = 0.0
        equity.append(prev_equity * (1 + ret))

    # ── 4. FORCE FINAL EXIT ───────────────────────────────────────────────
    if position != 0:
        p = price[-1]
        if position == 1:
            pnl, ttype = (p - entry_price) / entry_price - cost_pct, "LONG"
        else:
            pnl, ttype = (entry_price - p) / entry_price - cost_pct, "SHORT"
        trades.append({
            "type": ttype, "entry": entry_price, "exit": p,
            "pnl": pnl, "bars": n - entry_idx, "exit_reason": "forced",
        })

    equity = pd.Series(equity, index=df.index[:len(equity)])
    return equity, pd.DataFrame(trades)


def evaluate_performance(equity, trades):
    returns     = equity.pct_change().dropna()
    sharpe      = np.sqrt(252) * returns.mean() / (returns.std() + 1e-9)
    cum_return  = equity.iloc[-1] - 1
    drawdown    = equity / equity.cummax() - 1
    mdd         = drawdown.min()
    n_trades    = len(trades)
    win_rate    = (trades["pnl"] > 0).mean() if n_trades > 0 else 0.0
    avg_trade   = trades["pnl"].mean()        if n_trades > 0 else 0.0
    return {
        "sharpe": sharpe, "return": cum_return, "mdd": mdd,
        "trades": n_trades, "win_rate": win_rate, "avg_trade": avg_trade,
    }


def _empty_bt(tf, df):
    return {
        "timeframe": tf, "sharpe": 0.0, "max_drawdown": 0.0,
        "total_return": 0.0, "bnh_return": 0.0, "n_trades": 0,
        "win_rate": 0.0, "total_trade_cost": 0.0, "total_funding_cost": 0.0,
        "n_holdout_bars": 0, "df": df.iloc[0:0].copy(),
    }


def run_backtest_v15(df_feat, model, feature_cols, tf, train_median=None, scaler=None):

    df_bt = df_feat.copy()
    if len(df_bt) < 50:
        print(f"  {tf}: holdout too small ({len(df_bt)} bars) — skipping")
        return _empty_bt(tf, df_feat)

    train_med = train_median if train_median is not None else pd.Series(dtype=float)

    # Align features + impute with train median
    df_bt_feats = df_bt.reindex(columns=feature_cols, fill_value=np.nan)
    for col in feature_cols:
        if col in train_med.index:
            df_bt_feats[col] = df_bt_feats[col].fillna(train_med[col])
    df_bt_feats = df_bt_feats.replace([np.inf, -np.inf], 0).fillna(0)

    proba    = model.predict_proba(df_bt_feats.values)
    cls      = list(model.classes_)
    idx_sell = cls.index(0) if 0 in cls else None
    idx_buy  = cls.index(2) if 2 in cls else None
    p_sell   = proba[:, idx_sell] if idx_sell is not None else np.zeros(len(df_bt))
    p_buy    = proba[:, idx_buy]  if idx_buy  is not None else np.zeros(len(df_bt))

    # Directional edge: +ve = bullish, -ve = bearish
    edge     = p_buy - p_sell
    abs_edge = np.abs(edge)

    # Rolling percentile threshold — no look-ahead bias (falls back to global for first 200 bars)
    _rolling_thr = (
        pd.Series(abs_edge)
        .rolling(200, min_periods=20)
        .quantile(CONFIDENCE_PERCENTILE / 100.0)
        .fillna(np.percentile(abs_edge, CONFIDENCE_PERCENTILE))
        .values
    )
    threshold  = float(np.percentile(abs_edge, CONFIDENCE_PERCENTILE))
    total_cost = COST_PER_TRADE * 2

    adx_arr = (
        df_bt["adx_14"].fillna(25).values
        if "adx_14" in df_bt.columns
        else np.full(len(df_bt), 25.0)
    )
    # [v15-3] ATR for trailing stop; NaN → median fallback
    atr_col = "atr_14"
    if atr_col in df_bt.columns:
        atr_med = df_bt[atr_col].median()
        atr_arr = df_bt[atr_col].fillna(atr_med).values
    else:
        atr_arr = None

    # [v15-6] Volatility filter: atr_pct column if available
    atr_pct_arr = (
        df_bt["atr_pct"].fillna(0).values
        if "atr_pct" in df_bt.columns
        else np.full(len(df_bt), 1.0)
    )

    signal = pd.Series(1, index=df_bt.index)   # 1=hold, 0=sell, 2=buy
    last_trade_bar = -MIN_GAP_CANDLES - 1
    trades_today   = {}
    max_daily      = TF_MAX_TRADES_DAY.get(tf, 8)
    filter_counts  = {
        "edge_floor": 0, "threshold": 0, "cost": 0,
        "regime": 0, "gap": 0, "daily_cap": 0,
        "volatility": 0, "passed": 0,
    }

    for i in range(len(df_bt)):
        ae       = abs_edge[i]
        raw_edge = edge[i]
        adx_val  = adx_arr[i]
        vol_val  = atr_pct_arr[i]
        bar_date = df_bt["datetime"].iloc[i].date()

        if ae < EDGE_FLOOR:
            filter_counts["edge_floor"] += 1; continue
        if ae < _rolling_thr[i]:
            filter_counts["threshold"] += 1; continue
        if ae < total_cost * MIN_EDGE_COST_RATIO:
            filter_counts["cost"] += 1; continue
        if adx_val > 25.0:
            filter_counts["regime"] += 1; continue
        if i - last_trade_bar < MIN_GAP_CANDLES:
            filter_counts["gap"] += 1; continue
        if trades_today.get(bar_date, 0) >= max_daily:
            filter_counts["daily_cap"] += 1; continue
        # [v15-6] Skip low-volatility / dead-market conditions
        if vol_val < VOLATILITY_FILTER_PCT:
            filter_counts["volatility"] += 1; continue

        signal.iloc[i] = 2 if raw_edge > 0 else 0
        last_trade_bar = i
        trades_today[bar_date] = trades_today.get(bar_date, 0) + 1
        filter_counts["passed"] += 1

    print(f"  Filters     : {filter_counts}")
    print(f"  Edge thresh : {threshold:.4f}  floor: {EDGE_FLOOR:.4f}  "
          f"(top {100-CONFIDENCE_PERCENTILE}%)")

    df_bt["ml_signal"] = signal

    equity, trades_df = run_backtest(
        df_bt, signal, price_col="close",
        atr_arr=atr_arr, atr_mult=ATR_TRAILING_MULT,   # [v15-3]
        hard_sl_pct=0.01, cost_pct=COST_PER_TRADE,
        edge_arr=edge, edge_decay_thr=EDGE_DECAY_THRESHOLD,  # [v15-5]
        min_hold_bars=MIN_HOLD_BARS,                         # [v15-4]
    )

    df_bt["market_return"] = df_bt["close"].pct_change()
    df_bt["cum_market"]    = (1 + df_bt["market_return"]).cumprod()
    df_bt["cum_strategy"]  = equity
    df_bt["drawdown"]      = equity / equity.cummax() - 1

    perf          = evaluate_performance(equity, trades_df)
    sharpe        = perf["sharpe"]
    max_drawdown  = perf["mdd"]
    total_return  = perf["return"]
    n_trades      = perf["trades"]
    win_rate      = perf["win_rate"]
    bnh_return    = float(df_bt["cum_market"].iloc[-1] - 1)
    total_trade_cost = n_trades * COST_PER_TRADE if n_trades > 0 else 0

    # Exit reason breakdown
    if not trades_df.empty and "exit_reason" in trades_df.columns:
        print(f"  Exit reasons: {trades_df['exit_reason'].value_counts().to_dict()}")

    return {
        "timeframe":          tf,
        "sharpe":             round(sharpe, 4),
        "max_drawdown":       round(max_drawdown, 4),
        "total_return":       round(total_return, 4),
        "bnh_return":         round(bnh_return, 4),
        "n_trades":           n_trades,
        "win_rate":           round(win_rate, 4),
        "total_trade_cost":   round(total_trade_cost, 6),
        "total_funding_cost": 0.0,
        "n_holdout_bars":     len(df_bt),
        "df":                 df_bt,
        "trades":             trades_df,
    }


# ── Execute backtests ─────────────────────────────────────────────────────────
backtest_results = {}
for tf in TIMEFRAMES:
    res = training_results.get(tf, {})
    if not res or res.get("model") is None:
        print(f"\n  {tf}: no model — skipping backtest")
        continue

    print(f"\n{'='*60}\nBacktesting: {tf}")
    feature_cols  = training_results[tf]["features"]
    train_median_ = training_results[tf].get("train_median")

    bt = run_backtest_v15(
        holdout_splits[tf], res["model"], feature_cols, tf,
        train_median=train_median_
    )
    backtest_results[tf] = bt

    print(f"  Sharpe Ratio         : {bt['sharpe']:>8.4f}")
    print(f"  Max Drawdown         : {bt['max_drawdown']:>8.2%}")
    print(f"  Total Return         : {bt['total_return']:>8.2%}")
    print(f"  Buy-and-Hold Return  : {bt['bnh_return']:>8.2%}")
    print(f"  Win Rate             : {bt['win_rate']:>8.2%}")
    print(f"  Trades               : {bt['n_trades']:>8}")
    print(f"  Trade Cost (total)   : {bt['total_trade_cost']:>10.4%}")

    if "trades" in bt and not bt["trades"].empty:
        print("  Sample trades:")
        print(bt["trades"].head().to_string(index=False))

    if   bt["sharpe"] >= 1.5: print("  Sharpe >= 1.5 — institutional grade")
    elif bt["sharpe"] >= 1.0: print("  Sharpe >= 1.0 — strong; ready for paper trading")
    elif bt["sharpe"] >= 0.5: print("  Sharpe 0.5-1.0 — acceptable; monitor live")
    elif bt["sharpe"] >= 0:   print("  Sharpe 0-0.5 — marginal; tune EDGE_FLOOR or TOP_N")
    else:                     print("  Negative Sharpe — check label config or feature set")

    if res.get("dir"):
        meta_path = Path(res["dir"]) / f"metadata_BTCUSD_{tf}.json"
        if meta_path.exists():
            meta_d = json.loads(meta_path.read_text())
            meta_d["backtest"] = {k: v for k, v in bt.items() if k not in ["df", "trades"]}
            meta_path.write_text(json.dumps(meta_d, indent=2))

# ── Equity curves ─────────────────────────────────────────────────────────────
n_tf = len(TIMEFRAMES)
fig, axes = plt.subplots(n_tf, 1, figsize=(14, 4*n_tf), squeeze=False)
for ax, tf in zip(axes[:, 0], TIMEFRAMES):
    bt = backtest_results.get(tf)
    if bt is None or bt["df"].empty:
        ax.text(0.5, 0.5, f"{tf}: no data", transform=ax.transAxes, ha="center")
        continue
    df_bt = bt["df"]
    ax.plot(df_bt.index, df_bt["cum_market"],   label="Buy & Hold",      alpha=0.5, lw=1.2, color="gray")
    ax.plot(df_bt.index, df_bt["cum_strategy"], label="ML Strategy v15", lw=1.8,   color="steelblue")
    ax.fill_between(df_bt.index, df_bt["drawdown"]+1, 1, alpha=0.15, color="red", label="Drawdown")
    ax.set_title(
        f"{tf}  |  Sharpe: {bt['sharpe']:.2f}  MDD: {bt['max_drawdown']:.1%}  "
        f"Return: {bt['total_return']:.1%}  Trades: {bt['n_trades']}"
    )
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylabel("Cumulative Return")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "backtest_equity_curves_v15.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 7: SHAP feature importance -- v14
# [v14-1] SHAP drawn from train_splits (holdout stays unseen)
# [v14-3] Train median imputed before SHAP

for tf in TIMEFRAMES:
    res = training_results.get(tf, {})
    if not res or res.get("model") is None:
        print(f"{tf}: no model"); continue

    # [v14-1] Use TRAIN zone for SHAP — holdout stays clean
    df           = train_splits[tf]
    feature_cols = training_results[tf]["features"]
    train_med    = training_results[tf].get("train_median", pd.Series(dtype=float))
    X_df = df.reindex(columns=feature_cols, fill_value=np.nan)
    # [v14-3] Impute with train median
    if train_med is not None and len(train_med) > 0:
        X_df = X_df.fillna(train_med)
    X_s = X_df.replace([np.inf, -np.inf], 0).fillna(0).values

    sample_size = min(2000, len(X_s))  # [v13-8] increased from 500 for better SHAP coverage
    np.random.seed(RANDOM_STATE)
    X_sample = X_s[np.random.choice(len(X_s), sample_size, replace=False)]

    model = res["model"]   # XGBClassifier directly -- no unwrap needed

    try:
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
    except Exception as e:
        print(f"  SHAP failed for {tf}: {e}")
        continue

    if isinstance(shap_values, list):
        mean_abs_shap = np.mean([np.mean(np.abs(sv), axis=0) for sv in shap_values], axis=0)
    else:
        abs_s = np.abs(shap_values)
        mean_abs_shap = (np.mean(np.mean(abs_s, axis=0), axis=1)
                         if abs_s.ndim == 3 else np.mean(abs_s, axis=0))

    top_idx      = np.argsort(mean_abs_shap)[-20:][::-1]
    top_features = [(feature_cols[i], mean_abs_shap[i]) for i in top_idx]
    max_imp      = max(mean_abs_shap) + 1e-12

    print(f"\n{tf} -- Top 20 Features (mean |SHAP|):")
    for feat, imp in top_features:
        bar = "#" * int(imp / max_imp * 30)
        print(f"  {feat:<38} {imp:.5f}  {bar}")

print("\nSHAP analysis complete (v14).")


In [ ]:
# Cell 8: ModelRegistry + RetrainEngine — v16
# ═══════════════════════════════════════════════════════════════════════
#  [v16-1] ModelRegistry — versioned save / load / hot-swap
#  [v16-2] RetrainEngine — modular, fully self-contained retrain logic
#  [v16-3] Lightweight warm-start integrated (no CV)
#  [v16-4] Auto-apply: training_results updated automatically on accept
#  [v16-5] hot_reload_model() — drop-in for live agent integration
#  [v16-6] Cooldown tracked inside registry (no external log dependency)
#  [v16-8] Validation gate: checks both F1 delta AND win-rate delta
#
# ┌──────────────────────────────────────────────────────────────────┐
# │  FLOW (what this cell replaces):                                 │
# │  drift → retrain → validate(F1+winrate) → accept/reject         │
# │  → save versioned .pkl → update _latest.pkl → hot-swap model    │
# └──────────────────────────────────────────────────────────────────┘
#
# NOTE: Run AFTER main training + backtest. Not during backtesting.

import json, pickle, time
import xgboost as xgb
from sklearn.base import clone


# ═══════════════════════════════════════════════════════════════════════
#  ModelRegistry [v16-1]
# ═══════════════════════════════════════════════════════════════════════

class ModelRegistry:
    """
    Manages versioned model storage, latest-pointer updates, and loading.
    Keeps an in-memory registry so hot_reload_model() never hits disk
    more than once per version.
    """

    def __init__(self, base_dir: Path):
        self.base_dir   = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self._cache     = {}        # tf -> latest loaded pipeline dict
        self._retrain_ts = {}       # tf -> last retrain timestamp (epoch)
        self._log_path   = self.base_dir / "retrain_log.json"

    # ── Save ──────────────────────────────────────────────────────────
    def save_version(self, tf: str, pipeline: dict, tag: str) -> Path:
        """Save a versioned pipeline and update the latest pointer."""
        version_dir = self.base_dir / f"model_{tf}_{tag}"
        version_dir.mkdir(parents=True, exist_ok=True)
        versioned_path = version_dir / f"pipeline_{tf}_{tag}.pkl"
        latest_path    = self.base_dir / f"pipeline_{tf}_latest.pkl"
        with open(versioned_path, "wb") as fh:
            pickle.dump(pipeline, fh)
        with open(latest_path, "wb") as fh:
            pickle.dump(pipeline, fh)
        self._cache[tf] = pipeline
        self._retrain_ts[tf] = time.time()
        print(f"  [Registry] Saved {tf} version={tag}")
        print(f"  [Registry] Latest pointer → pipeline_{tf}_latest.pkl")
        return versioned_path

    # ── Load ──────────────────────────────────────────────────────────
    def load_latest(self, tf: str) -> dict | None:
        """Load pipeline from the _latest.pkl pointer (with in-memory cache)."""
        if tf in self._cache:
            return self._cache[tf]
        latest_path = self.base_dir / f"pipeline_{tf}_latest.pkl"
        if not latest_path.exists():
            print(f"  [Registry] No latest pipeline for {tf}")
            return None
        with open(latest_path, "rb") as fh:
            pipeline = pickle.load(fh)
        self._cache[tf] = pipeline
        return pipeline

    def load_version(self, tf: str, tag: str) -> dict | None:
        """Load a specific versioned pipeline (for rollback)."""
        p = self.base_dir / f"model_{tf}_{tag}" / f"pipeline_{tf}_{tag}.pkl"
        if not p.exists():
            print(f"  [Registry] Version {tag} not found for {tf}")
            return None
        with open(p, "rb") as fh:
            return pickle.load(fh)

    def rollback(self, tf: str, tag: str) -> bool:
        """Restore a specific version as the current latest."""
        pipeline = self.load_version(tf, tag)
        if pipeline is None:
            return False
        latest_path = self.base_dir / f"pipeline_{tf}_latest.pkl"
        with open(latest_path, "wb") as fh:
            pickle.dump(pipeline, fh)
        self._cache[tf] = pipeline
        print(f"  [Registry:{tf}] Rolled back to version {tag}")
        return True

    # ── Cooldown ──────────────────────────────────────────────────────
    def cooldown_remaining(self, tf: str, hours: float) -> float:
        """Return hours remaining on cooldown, 0 if expired."""
        last = self._retrain_ts.get(tf, 0.0)
        if not last:  # check log file for persistence across restarts
            last = self._last_retrain_from_log(tf)
        if not last:
            return 0.0
        elapsed = (time.time() - last) / 3600
        return max(0.0, hours - elapsed)

    # ── Log ───────────────────────────────────────────────────────────
    def log_retrain(self, entry: dict):
        entries = self._load_log()
        entries.append(entry)
        with open(self._log_path, "w") as fh:
            json.dump(entries, fh, indent=2)

    def _load_log(self) -> list:
        if not self._log_path.exists():
            return []
        try:
            with open(self._log_path) as fh:
                return json.load(fh)
        except Exception:
            return []

    def _last_retrain_from_log(self, tf: str) -> float:
        entries = [e for e in self._load_log() if e.get("tf") == tf]
        if not entries:
            return 0.0
        try:
            return pd.Timestamp(entries[-1]["time"]).timestamp()
        except Exception:
            return 0.0

    def list_versions(self, tf: str) -> list:
        return sorted([d.name for d in self.base_dir.iterdir()
                       if d.is_dir() and d.name.startswith(f"model_{tf}_")])

    def invalidate_cache(self, tf: str):
        self._cache.pop(tf, None)


# ═══════════════════════════════════════════════════════════════════════
#  hot_reload_model [v16-5]
# ═══════════════════════════════════════════════════════════════════════

def hot_reload_model(tf: str, registry: "ModelRegistry") -> dict | None:
    """
    Drop-in for live agents: always returns the latest accepted pipeline.
    Usage in your agent:
        pipeline = hot_reload_model("5m", model_registry)
        model    = pipeline["model"]
        features = pipeline["features"]
        median   = pipeline["train_median"]
    """
    registry.invalidate_cache(tf)   # force re-read from disk
    pipeline = registry.load_latest(tf)
    if pipeline is None:
        print(f"  [HotReload:{tf}] No pipeline on disk — using in-memory model")
        return None
    version = pipeline.get("meta", {}).get("model_version", "?")
    retrained = pipeline.get("meta", {}).get("retrained_at", "")
    print(f"  [HotReload:{tf}] Loaded version={version}"
          + (f"  retrained_at={retrained[:19]}" if retrained else ""))
    return pipeline


# ═══════════════════════════════════════════════════════════════════════
#  RetrainEngine [v16-2]
# ═══════════════════════════════════════════════════════════════════════

class RetrainEngine:
    """
    Self-contained adaptive retrain engine.
    Call maybe_retrain() from any scheduler / agent / notebook cell.

    Parameters
    ----------
    registry           : ModelRegistry — handles versioning/saving
    training_results   : dict          — in-memory model store (auto-updated)
    labeled_data       : dict          — full labeled dataset per tf
    cooldown_hours     : float         — min hours between retrains
    drift_feature_limit: int           — trigger retrain if > N features drift
    validation_min_f1_delta: float     — new_f1 must be >= old_f1 + delta
    """

    def __init__(
        self,
        registry: ModelRegistry,
        training_results: dict,
        labeled_data: dict,
        cooldown_hours: float = RETRAIN_COOLDOWN_HOURS,
        drift_feature_limit: int = DRIFT_FEATURE_LIMIT,
        validation_min_f1_delta: float = 0.0,   # [v16-8] can require improvement margin
    ):
        self.registry     = registry
        self.tr           = training_results
        self.ld           = labeled_data
        self.cooldown_h   = cooldown_hours
        self.drift_limit  = drift_feature_limit
        self.min_f1_delta = validation_min_f1_delta

    # ── Public entry point ────────────────────────────────────────────
    def maybe_retrain(self, tf: str) -> bool:
        """
        Run the full retrain pipeline for a given timeframe.
        Returns True if a new model was accepted and deployed.
        """
        print(f"\n[RetrainEngine:{tf}] Starting adaptive check")

        if not self._has_base_model(tf):
            return False

        remaining = self.registry.cooldown_remaining(tf, self.cooldown_h)
        if remaining > 0:
            print(f"  Cooldown: {remaining:.1f}h remaining — skip")
            return False

        df           = self.ld.get(tf, pd.DataFrame())
        feature_cols = self.tr[tf]["features"]

        if len(df) < 40_000:
            print(f"  Insufficient data ({len(df):,} rows) — skip")
            return False

        # Step 1 — Drift detection
        drifted = self._detect_drift(df, feature_cols)
        print(f"  Drifted features: {len(drifted)}"
              + (f" {[d[0] for d in drifted[:3]]}..." if len(drifted) > 3 else
                 f" {[d[0] for d in drifted]}"))

        if len(drifted) <= self.drift_limit:
            print(f"  Drift ({len(drifted)}) <= limit ({self.drift_limit}) — no retrain needed")
            return False

        print(f"  Drift limit exceeded — RETRAINING")

        # Step 2 — Prepare training window
        df_clean = self._prepare_window(df, feature_cols)
        if df_clean is None:
            return False

        # Step 3 — Warm-start retrain
        new_model, new_median_dict, f1_train = self._warm_retrain(tf, df_clean, feature_cols)

        # Step 4 — Validation gate [v16-8]
        accepted = self._validate(tf, df_clean, feature_cols, new_model)
        if not accepted:
            return False

        old_f1, new_f1, old_wr, new_wr = accepted

        # Step 5 — Save + hot-swap
        version_tag = f"v_auto_{int(time.time())}"
        pipeline = {
            "model":        new_model,
            "features":     feature_cols,
            "train_median": new_median_dict,
            "meta": {
                **self.tr[tf].get("meta", {}),
                "retrained_at": pd.Timestamp.now("UTC").isoformat(),
                "retrain_method": "warm_start",
                "boost_rounds_added": NUM_BOOST_ROUND_INCREMENTAL,
                "n_samples": len(df_clean),
                "new_f1": float(new_f1),
                "old_f1": float(old_f1),
                "new_win_rate": float(new_wr),
                "old_win_rate": float(old_wr),
            },
        }
        self.registry.save_version(tf, pipeline, version_tag)

        # [v16-4] Auto-apply to in-memory training_results (no manual copy-paste)
        self.tr[tf]["model"]        = new_model
        self.tr[tf]["train_median"] = pd.Series(new_median_dict)
        self.tr[tf]["mean_f1"]      = float(new_f1)
        if "meta" not in self.tr[tf]:
            self.tr[tf]["meta"] = {}
        self.tr[tf]["meta"].update(pipeline["meta"])

        self.registry.log_retrain({
            "tf": tf,
            "time": pd.Timestamp.now("UTC").isoformat(),
            "version": version_tag,
            "drifted": len(drifted),
            "n_samples": len(df_clean),
            "old_f1": round(float(old_f1), 4),
            "new_f1": round(float(new_f1), 4),
            "old_win_rate": round(float(old_wr), 4),
            "new_win_rate": round(float(new_wr), 4),
        })

        print(f"  ✅ ACCEPTED  version={version_tag}")
        print(f"     F1: {old_f1:.4f} → {new_f1:.4f} | WinRate: {old_wr:.3f} → {new_wr:.3f}")
        print(f"     training_results['{tf}'] updated automatically [v16-4]")
        return True

    # ── Internal helpers ──────────────────────────────────────────────
    def _has_base_model(self, tf):
        res = self.tr.get(tf, {})
        if not res or res.get("model") is None:
            print(f"  No base model for {tf} — skip")
            return False
        return True

    def _detect_drift(self, df, feature_cols):
        drifted = []
        df_recent = df.iloc[-20_000:]
        df_past   = df.iloc[-40_000:-20_000]
        for col in feature_cols:
            if col not in df.columns:
                continue
            a = df_past[col].dropna().values
            b = df_recent[col].dropna().values
            if len(a) < 30 or len(b) < 30:
                continue
            stat, p = ks_2samp(a, b)
            if p < DRIFT_ALPHA and stat > DRIFT_STAT_THRESHOLD:
                drifted.append((col, round(p, 6), round(stat, 4)))
        return drifted

    def _prepare_window(self, df, feature_cols):
        cutoff   = df["datetime"].max() - pd.Timedelta(days=ROLLING_DAYS)
        df_train = df[df["datetime"] >= cutoff].copy()
        df_clean = df_train[feature_cols + ["label"]].copy()
        df_clean[feature_cols] = (df_clean[feature_cols]
                                   .replace([np.inf, -np.inf], np.nan)
                                   .ffill(limit=5))
        df_clean = df_clean.dropna(subset=["label"])
        if len(df_clean) < 1_000:
            print(f"  Recent window too small ({len(df_clean):,} rows) — skip")
            return None
        return df_clean

    def _warm_retrain(self, tf, df_clean, feature_cols):
        old_model = self.tr[tf]["model"]
        median_s  = df_clean[feature_cols].median()
        X         = df_clean[feature_cols].fillna(median_s).values
        y         = df_clean["label"].values
        sw        = np.array([CLASS_WEIGHTS.get(int(lbl), 1.0) for lbl in y])

        old_booster   = old_model.get_booster()
        native_params = dict(old_model.get_xgb_params())
        native_params["num_class"] = 3
        dtrain    = xgb.DMatrix(X, label=y, weight=sw)
        new_boost = xgb.train(native_params, dtrain,
                              num_boost_round=NUM_BOOST_ROUND_INCREMENTAL,
                              xgb_model=old_booster, verbose_eval=False)

        new_model = clone(old_model)
        new_model._Booster = new_boost
        for attr in ("classes_", "_le", "n_features_in_", "_features_count"):
            if hasattr(old_model, attr):
                try: setattr(new_model, attr, getattr(old_model, attr))
                except Exception: pass
        if not hasattr(new_model, "n_classes_"):
            new_model.n_classes_ = int(native_params.get("num_class", 3))

        f1 = f1_score(y, new_model.predict(X), average="macro", zero_division=0)
        print(f"  Warm-start retrain: F1_train={f1:.4f}  Samples={len(y):,}")
        return new_model, median_s.to_dict(), f1

    def _validate(self, tf, df_clean, feature_cols, new_model):
        """
        Validation gate [v16-8]:
        Accept only if new_f1 >= old_f1 + min_f1_delta
        AND new win_rate >= old win_rate - 0.02 (tolerates small drops).
        Returns (old_f1, new_f1, old_win_rate, new_win_rate) or None if rejected.
        """
        old_model = self.tr[tf]["model"]
        prod_med  = self.tr[tf].get("train_median")
        if hasattr(prod_med, "to_dict"):
            prod_med = prod_med.to_dict()
        prod_med  = prod_med or {}

        n_val = min(VALIDATION_WINDOW_ROWS, len(df_clean))
        df_val = df_clean.iloc[-n_val:].dropna(subset=["label"])
        if len(df_val) < MIN_VAL_ROWS:
            print(f"  Validation set too small ({len(df_val)}) — REJECT")
            return None

        X_val = df_val[feature_cols].fillna(prod_med).values
        y_val = df_val["label"].values

        old_pred = old_model.predict(X_val)
        new_pred = new_model.predict(X_val)

        old_f1  = f1_score(y_val, old_pred, average="macro", zero_division=0)
        new_f1  = f1_score(y_val, new_pred, average="macro", zero_division=0)

        # Win-rate proxy: fraction of non-HOLD predictions that are correct
        def _win_rate(y_true, y_pred):
            mask = y_pred != 1   # non-HOLD signals
            if mask.sum() == 0:
                return 0.5
            return (y_true[mask] == y_pred[mask]).mean()

        old_wr = _win_rate(y_val, old_pred)
        new_wr = _win_rate(y_val, new_pred)

        print(f"  Validation — old_F1={old_f1:.4f}  new_F1={new_f1:.4f}"
              f"  old_WR={old_wr:.3f}  new_WR={new_wr:.3f}")

        # Gate conditions
        if new_f1 < old_f1 + self.min_f1_delta:
            print(f"  REJECTED — F1 did not improve by {self.min_f1_delta:.4f}")
            return None
        if new_wr < old_wr - 0.02:
            print(f"  REJECTED — Win-rate dropped more than 2pp")
            return None

        return old_f1, new_f1, old_wr, new_wr


# ═══════════════════════════════════════════════════════════════════════
#  Initialise registry and run adaptive engine
# ═══════════════════════════════════════════════════════════════════════

model_registry = ModelRegistry(OUTPUT_DIR)

# Register existing models in the registry cache (from Cell 5)
for tf, res in training_results.items():
    if res.get("model") is not None:
        pipeline = {
            "model":        res["model"],
            "features":     res["features"],
            "train_median": (res["train_median"].to_dict()
                             if hasattr(res.get("train_median"), "to_dict")
                             else res.get("train_median", {})),
            "meta":         res.get("meta", {}),
        }
        model_registry._cache[tf] = pipeline

retrain_engine = RetrainEngine(
    registry              = model_registry,
    training_results      = training_results,
    labeled_data          = labeled_data,
    cooldown_hours        = RETRAIN_COOLDOWN_HOURS,
    drift_feature_limit   = DRIFT_FEATURE_LIMIT,
    validation_min_f1_delta = 0.0,   # set to 0.005 to require measurable improvement
)

print("=" * 60)
print("ADAPTIVE ENGINE v16 — ModelRegistry + RetrainEngine")
print("=" * 60)

adaptive_triggered = {}
for tf in TIMEFRAMES:
    triggered = retrain_engine.maybe_retrain(tf)
    adaptive_triggered[tf] = triggered
    if not triggered:
        print(f"  [{tf}] No adaptive update triggered — model unchanged")

print(f"\nAdaptive results: {adaptive_triggered}")
print("\nAvailable registry versions:")
for tf in TIMEFRAMES:
    versions = model_registry.list_versions(tf)
    print(f"  {tf}: {versions}")

print("\n─── Hot-reload test ───────────────────────────────────────")
for tf in TIMEFRAMES:
    pipeline = hot_reload_model(tf, model_registry)
    if pipeline:
        meta = pipeline.get("meta", {})
        print(f"  {tf}: version={meta.get('model_version','?')}  "
              f"features={len(pipeline.get('features',[]))}")

print("\n─── Agent integration snippet ─────────────────────────────")
print("""
# In your live trading agent (retrain_engine.py):
from pathlib import Path
import pickle

OUTPUT_DIR = Path('/content/drive/MyDrive/JackSparrow_Models')

def load_live_model(tf):
    path = OUTPUT_DIR / f'pipeline_{tf}_latest.pkl'
    with open(path, 'rb') as fh:
        return pickle.load(fh)

# Every N hours, check and retrain:
# retrain_engine.maybe_retrain("5m")
# retrain_engine.maybe_retrain("15m")
#
# When a signal fires, always hot-reload:
# pipeline = hot_reload_model("5m", model_registry)
""")


In [ ]:
# Cell 9: Final validation summary — v16
print("\n" + "=" * 60)
print("FINAL VALIDATION SUMMARY  (JackSparrow v16)")
print("=" * 60)
print("Signal : edge = p_buy - p_sell | XGBoost | balanced weights | no SMOTE")
print("Exits  : ATR trailing | hard SL | edge decay | min hold")
print("Lifecycle: ModelRegistry + RetrainEngine | versioned + hot-swap")
print("=" * 60)

all_pass = True

for tf in TIMEFRAMES:
    res = training_results.get(tf, {})
    bt  = backtest_results.get(tf, {})

    if not res or res.get("model") is None:
        print(f"\n  {tf}: NO MODEL TRAINED")
        print(f"    >> Check Cell 2.5 — if coverage < 90%, fix Cell 2 fetch.")
        all_pass = False
        continue

    n_train   = len(train_splits.get(tf, pd.DataFrame()))
    n_holdout = len(holdout_splits.get(tf, pd.DataFrame()))
    mean_f1   = res.get("mean_f1", 0)
    sharpe    = bt.get("sharpe")
    mdd       = bt.get("max_drawdown")
    ret       = bt.get("total_return")
    n_trades  = bt.get("n_trades")
    win_rate  = bt.get("win_rate")
    n_feat    = len(res.get("features", []))

    # Check if an adaptive update was accepted [v16-4]
    was_updated = adaptive_triggered.get(tf, False)
    model_tag   = res.get("meta", {}).get("model_version", "v16")
    retrained_at = res.get("meta", {}).get("retrained_at", "")

    print(f"\n  {tf}:")
    print(f"    model_version        : {model_tag}" + (" [ADAPTIVE UPDATE APPLIED]" if was_updated else ""))
    if retrained_at:
        print(f"    retrained_at         : {retrained_at[:19]} UTC")
    print(f"    features selected    : {n_feat}")
    print(f"    training rows        : {n_train:,}  (holdout: {n_holdout:,} rows)")
    print(f"    cv_f1_macro (honest) : {mean_f1:.4f}  [TimeSeriesSplit, no leakage]")
    print(f"    backtest.sharpe      : {sharpe}")
    print(f"    backtest.mdd         : {mdd:.2%}" if mdd is not None else "    backtest.mdd : N/A")
    print(f"    backtest.return      : {ret:.2%}" if ret is not None else "    backtest.return : N/A")
    print(f"    backtest.trades      : {n_trades}")
    print(f"    backtest.win_rate    : {win_rate:.2%}" if win_rate is not None else "    backtest.win_rate : N/A")

    registry_versions = model_registry.list_versions(tf)
    print(f"    registry versions    : {registry_versions}")

    if sharpe is None:
        print("    WARN: No backtest result")
    elif sharpe >= 1.5: print("    PASS: Sharpe >= 1.5 — institutional grade")
    elif sharpe >= 1.0: print("    PASS: Sharpe >= 1.0 — strong; ready for paper trading")
    elif sharpe >= 0.5: print("    PASS: Sharpe 0.5-1.0 — acceptable; monitor live")
    elif sharpe >= 0:   print("    MARGINAL: Sharpe 0-0.5 — tune EDGE_FLOOR or TOP_N_FEATURES"); all_pass = False
    else:               print("    FAIL: Negative Sharpe — review label config or feature set"); all_pass = False

    if n_train < 10_000:
        print(f"    WARN: Low training rows ({n_train:,})")
        all_pass = False

print(f"\n{'=' * 60}")
if all_pass:
    print("ALL TIMEFRAMES VALIDATED — ready for paper trading")
    print(f"   Output: {OUTPUT_DIR}")
else:
    print("REVIEW REQUIRED — address warnings above before going live")

print(f"\nActive timeframes : {TIMEFRAMES}")
print("Reminder: Paper-trade >= 1 week before live capital")
print("Reminder: 5x isolated margin on Delta Exchange India")
print("\nQuick-tune knobs (Cell 1):")
print("  EDGE_FLOOR            (default 0.15) — raise to reduce trades, improve quality")
print("  CONFIDENCE_PERCENTILE (default 90)   — raise for even stronger signal filter")
print("  TOP_N_FEATURES        (default 20)   — lower to reduce noise")
print("  ATR_TRAILING_MULT     (default 1.5)  — raise for wider stop, lower for tighter")
print("  MIN_HOLD_BARS         (default 5)    — raise to prevent more premature exits")
print("  EDGE_DECAY_THRESHOLD  (default 0.05) — raise to exit sooner on fading edge")
print("  VOLATILITY_FILTER_PCT (default 0.002)— raise to trade only in active markets")
print("  RETRAIN_LIGHTWEIGHT   (default False) — set True for fast warm-start retrains")
print("  RETRAIN_COOLDOWN_HOURS(default 12)   — minimum hours between retrains")


In [ ]:
# Cell 10: Download model artifacts — v16
# [v16-1] Now downloads versioned .pkl files AND latest pointer files.
# [v16]   Also includes retrain_log.json for audit trail.
#
# Options:
#   BROWSER_DOWNLOAD — trigger google.colab.files.download (your PC's Downloads).
#   ALSO_COPY_ZIP_TO — copy the same zip to Drive or another path (e.g. persistent backup).

BROWSER_DOWNLOAD = True
ALSO_COPY_ZIP_TO = None  # e.g. Path("/content/drive/MyDrive/JackSparrow_Models_v16.zip")

import shutil, tempfile
from pathlib import Path

temp_dir  = tempfile.mkdtemp()
temp_path = Path(temp_dir)

print(f"Selective download: v16 models for {TIMEFRAMES}")

copied_items = []

for tf in TIMEFRAMES:
    # Primary: v16 directory
    for tag in ["v16", "v14"]:
        src_dir = OUTPUT_DIR / f"model_{tf}_{tag}"
        if src_dir.exists():
            dst_dir = temp_path / f"model_{tf}_{tag}"
            shutil.copytree(src_dir, dst_dir)
            copied_items.append(f"model_{tf}_{tag}")
            print(f"  ✓ Copied: {src_dir.name}")
            break

    # Copy any auto-retrain versions
    for version_dir in OUTPUT_DIR.glob(f"model_{tf}_v_auto_*"):
        dst_dir = temp_path / version_dir.name
        if not dst_dir.exists():
            shutil.copytree(version_dir, dst_dir)
            copied_items.append(version_dir.name)
            print(f"  ✓ Copied adaptive version: {version_dir.name}")

    # Copy latest pointer [v16-1]
    latest = OUTPUT_DIR / f"pipeline_{tf}_latest.pkl"
    if latest.exists():
        shutil.copy2(latest, temp_path / latest.name)
        copied_items.append(latest.name)
        print(f"  ✓ Copied: {latest.name}")

# Equity curve plot
for plot_name in ["backtest_equity_curves_v15.png", "backtest_equity_curves_v14.png"]:
    plot_src = OUTPUT_DIR / plot_name
    if plot_src.exists():
        shutil.copy2(plot_src, temp_path / plot_name)
        copied_items.append(plot_name)
        print(f"  ✓ Copied: {plot_name}")
        break

# Retrain log [v16]
log_src = OUTPUT_DIR / "retrain_log.json"
if log_src.exists():
    shutil.copy2(log_src, temp_path / "retrain_log.json")
    copied_items.append("retrain_log.json")
    print(f"  ✓ Copied: retrain_log.json")

if not copied_items:
    print("  ❌ No models found to download!")
else:
    print(f"  📦 Items ({len(copied_items)}): {copied_items}")
    zip_base = "/content/JackSparrow_Models_v16"
    zip_path = zip_base + ".zip"
    shutil.make_archive(zip_base, "zip", root_dir=temp_dir, base_dir=".")
    print(f"  Archive path (copy from Colab Files if needed): {zip_path}")
    if ALSO_COPY_ZIP_TO is not None:
        dest = Path(ALSO_COPY_ZIP_TO)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, dest)
        print(f"  Copied zip to: {dest}")
    try:
        if BROWSER_DOWNLOAD:
            from google.colab import files
            files.download(zip_path)
            print("  Browser download triggered.")
        else:
            print("  BROWSER_DOWNLOAD=False — use Archive path above or ALSO_COPY_ZIP_TO.")
    except ImportError:
        print(f"  Not in Colab. Archive at: {zip_path}")

shutil.rmtree(temp_dir)
print("Temp directory cleaned up.")
